# CPS `fact_enrollment_annualized` — Data Cleaning & Deduplication

**Project:** Chicago Public Schools Enrollment Forecasting (ML Pipeline)  
**Input:** `fact_enrollment_annualized` (Fabric Lakehouse — raw/uncleaned)  
**Output:** `fact_enrollment_annualized_clean` (Fabric Lakehouse — deduplicated, validated, ML-ready)  

---


The EDA notebook revealed **severe data quality issues** in the raw `fact_enrollment_annualized` table that would corrupt any downstream ML model if left unaddressed:

| Problem | Impact | Scale |
|---|---|---|
| Every `ENROLLMENT_ANNUALIZED_KEY` appears exactly **2 times** | Inflates all enrollment counts by 2x. An ETL/pipeline load duplication — not a business event. | **9,875,844** duplicate rows (50% of the table) |
| `SYS_DELETE_STATUS = 1` (soft-deleted records still in table) | Deleted records participate in counts and aggregations | **272** rows |
| `SCHOOL_YEAR` stored as `double` (e.g. 2024.0 instead of 2024) | Join failures with dimension tables; display issues; floating-point comparison risk | All rows |
| Sentinel dates `9999-12-31` in `SOURCE_ENROLLMENT_EXIT_DATE` | Duration calculations produce absurd values (millions of days) | **2,718,858** rows (13.77%) |
| `SOURCE_ENROLLMENT_ENTRY_DATE > SOURCE_ENROLLMENT_EXIT_DATE` | Logically impossible — corrupts enrollment duration features | **10** rows |
| `ENTRY_REASON` is 97.33% null, `EXIT_REASON` is 89.24% null | These columns carry almost no information; waste compute | All rows |
| Future school year records (SY ≥ 2027) | Pre-enrolled/projected students mixed with actuals — would leak future data into training | **663,552** rows |
| 69.65% of records have `ENROLLMENT_DAYS = 0` | These are annualized tenure records where school hasn't started in that window | **13,756,948** rows |

### What This Notebook Does (In Order)

1. **Load & baseline metrics** — record the raw state for audit trail  
2. **Remove exact row duplicates** — drop the ETL-caused 2x duplication  
3. **Remove soft-deleted records** — filter `SYS_DELETE_STATUS = 1`  
4. **Fix data types** — cast `SCHOOL_YEAR` from double to integer  
5. **Handle sentinel dates** — replace `9999-12-31` with NULL  
6. **Remove logically invalid records** — entry date > exit date  
7. **Drop near-empty columns** — `ENTRY_REASON`, `EXIT_REASON`  
8. **Add cleaning metadata columns** — flags for downstream filtering  
9. **Validate & write** — final quality checks, write clean table to Lakehouse  


---
## 1. Setup & Load Raw Data

In [30]:
# ============================================================
# 1.1 — Imports
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DateType
from pyspark.sql.window import Window
import datetime

print("✔ Imports complete")

StatementMeta(, a92cb0f9-a6af-4b67-8c11-7f99f7dfaabc, 32, Finished, Available, Finished, False)

✔ Imports complete


In [31]:
# ============================================================
# 1.2 — Load raw table from Lakehouse
# ============================================================
RAW_TABLE = "fact_enrollment_annualized"  # Update to your full table path if needed

df_raw = spark.read.table(RAW_TABLE)

RAW_ROW_COUNT = df_raw.count()
RAW_COL_COUNT = len(df_raw.columns)

print(f"✔ Loaded '{RAW_TABLE}'")
print(f"  Raw rows    : {RAW_ROW_COUNT:,}")
print(f"  Raw columns : {RAW_COL_COUNT}")
print(f"  Unique keys : {df_raw.select('ENROLLMENT_ANNUALIZED_KEY').distinct().count():,}")

StatementMeta(, a92cb0f9-a6af-4b67-8c11-7f99f7dfaabc, 33, Finished, Available, Finished, False)

✔ Loaded 'fact_enrollment_annualized'
  Raw rows    : 19,751,688
  Raw columns : 29
  Unique keys : 9,875,844


---
## 2. Remove Exact Row Duplicates

### Why we are doing this

The EDA revealed that **every single `ENROLLMENT_ANNUALIZED_KEY` appears exactly 2 times** in the table, with all 29 columns being identical across both copies. This is **not** a business event (it's not a student enrolling twice). It is an **ETL/pipeline duplication** — the data was loaded twice, likely during a Fabric pipeline run or a source system extract that was replayed.

**Evidence from EDA:**
- Total rows: 19,751,688
- Distinct `ENROLLMENT_ANNUALIZED_KEY`: 9,875,844
- Maximum row count per key: 2 (no key appears 3+ times)
- Full-row duplicate count: 9,875,844 (exactly half the table)
- Records-per-student ratio was ~2.0 uniformly across all school years (2001–2024), confirming this is systematic, not random

**Action:** Use `dropDuplicates()` on ALL columns to remove exact copies. This is the safest dedup approach since it only removes rows where every single column matches — no business data is lost.

In [32]:
# ============================================================
# 2.1 — Pre-dedup validation: confirm the duplication pattern
# ============================================================
key_count_dist = (
    df_raw
    .groupBy("ENROLLMENT_ANNUALIZED_KEY")
    .count()
    .groupBy("count")
    .agg(F.count("*").alias("num_keys"))
    .orderBy("count")
)

print("Distribution of row counts per ENROLLMENT_ANNUALIZED_KEY:")
print("(We expect all keys to appear exactly 2 times)\n")
display(key_count_dist)

StatementMeta(, a92cb0f9-a6af-4b67-8c11-7f99f7dfaabc, 34, Finished, Available, Finished, False)

Distribution of row counts per ENROLLMENT_ANNUALIZED_KEY:
(We expect all keys to appear exactly 2 times)



SynapseWidget(Synapse.DataFrame, bdab6c7e-7ba1-4190-9f7d-880941ea5ba5)

In [33]:
# ============================================================
# 2.2 — Drop exact row duplicates
# ============================================================
df_deduped = df_raw.dropDuplicates()

deduped_count = df_deduped.count()
rows_removed_dedup = RAW_ROW_COUNT - deduped_count

print(f"Deduplication results:")
print(f"  Before : {RAW_ROW_COUNT:,}")
print(f"  After  : {deduped_count:,}")
print(f"  Removed: {rows_removed_dedup:,} ({rows_removed_dedup / RAW_ROW_COUNT * 100:.1f}%)")

StatementMeta(, a92cb0f9-a6af-4b67-8c11-7f99f7dfaabc, 35, Finished, Available, Finished, False)

Deduplication results:
  Before : 19,751,688
  After  : 9,875,844
  Removed: 9,875,844 (50.0%)


In [34]:
# ============================================================
# 2.3 — Post-dedup validation: ENROLLMENT_ANNUALIZED_KEY should now be unique
# ============================================================
post_dedup_keys = df_deduped.select("ENROLLMENT_ANNUALIZED_KEY").distinct().count()
remaining_dupes = deduped_count - post_dedup_keys

if remaining_dupes == 0:
    print(f"✔ ENROLLMENT_ANNUALIZED_KEY is now unique. {post_dedup_keys:,} distinct keys = {deduped_count:,} rows.")
else:
    print(f"⚠ {remaining_dupes:,} keys still have duplicates after dropDuplicates().")
    print("  This means some rows share the same key but differ in other columns.")
    print("  Investigating...")
    
    # Show which keys still have duplicates and which columns differ
    still_duped_keys = (
        df_deduped
        .groupBy("ENROLLMENT_ANNUALIZED_KEY")
        .count()
        .filter(F.col("count") > 1)
        .orderBy(F.desc("count"))
    )
    print(f"\n  Keys with remaining duplicates: {still_duped_keys.count():,}")
    display(still_duped_keys.limit(10))

StatementMeta(, a92cb0f9-a6af-4b67-8c11-7f99f7dfaabc, 36, Finished, Available, Finished, False)

✔ ENROLLMENT_ANNUALIZED_KEY is now unique. 9,875,844 distinct keys = 9,875,844 rows.


In [35]:
# ============================================================
# 2.4 — Handle near-duplicates (same key, different column values)
# If dropDuplicates() didn't fully resolve uniqueness, it means
# some ENROLLMENT_ANNUALIZED_KEYs have rows that differ in
# metadata columns (CREATED_TS, LAST_UPDATED_TS, etc.).
# Strategy: Keep the LATEST version per key (most recent LAST_UPDATED_TS).
# ============================================================

# Check if we still have duplicates after exact dedup
if remaining_dupes > 0:
    print("Resolving near-duplicates: keeping the latest version per ENROLLMENT_ANNUALIZED_KEY...")
    
    w = Window.partitionBy("ENROLLMENT_ANNUALIZED_KEY").orderBy(F.desc("LAST_UPDATED_TS"))
    
    df_deduped = (
        df_deduped
        .withColumn("_row_num", F.row_number().over(w))
        .filter(F.col("_row_num") == 1)
        .drop("_row_num")
    )
    
    final_dedup_count = df_deduped.count()
    extra_removed = deduped_count - final_dedup_count
    deduped_count = final_dedup_count
    rows_removed_dedup += extra_removed
    
    print(f"  Near-duplicates removed: {extra_removed:,}")
    print(f"  Final row count: {deduped_count:,}")
    
    # Verify uniqueness
    final_keys = df_deduped.select("ENROLLMENT_ANNUALIZED_KEY").distinct().count()
    assert final_keys == deduped_count, f"Key uniqueness FAILED: {final_keys} keys vs {deduped_count} rows"
    print(f"  ✔ ENROLLMENT_ANNUALIZED_KEY is now unique.")
else:
    print("✔ No near-duplicates. Exact dedup was sufficient.")

StatementMeta(, a92cb0f9-a6af-4b67-8c11-7f99f7dfaabc, 37, Finished, Available, Finished, False)

✔ No near-duplicates. Exact dedup was sufficient.


---
## 3. Remove Soft-Deleted Records

### Why we are doing this

The `SYS_DELETE_STATUS` column is a **soft-delete flag** used by the source system. Rows with `SYS_DELETE_STATUS = 1` have been logically deleted upstream — the enrollment was voided, corrected, or retracted. These rows should **never** participate in enrollment counts, feature engineering, or model training.

**Evidence from EDA:** 272 rows had `SYS_DELETE_STATUS = 1` (after deduplication this becomes ~136). Small in volume, but leaving them in would mean counting students who were administratively removed from the enrollment record.

In [36]:
# ============================================================
# 3.1 — Check SYS_DELETE_STATUS distribution
# ============================================================
print("SYS_DELETE_STATUS distribution (after dedup):")
display(
    df_deduped
    .groupBy("SYS_DELETE_STATUS")
    .agg(
        F.count("*").alias("row_count"),
        F.round(F.count("*") / deduped_count * 100, 4).alias("pct")
    )
    .orderBy("SYS_DELETE_STATUS")
)

StatementMeta(, a92cb0f9-a6af-4b67-8c11-7f99f7dfaabc, 38, Finished, Available, Finished, False)

SYS_DELETE_STATUS distribution (after dedup):


SynapseWidget(Synapse.DataFrame, 0ff94d12-014b-4bd3-a34a-622ed5ade6e1)

In [37]:
# ============================================================
# 3.2 — Filter out soft-deleted rows
# ============================================================
before_delete_filter = df_deduped.count()
df_clean = df_deduped.filter(F.col("SYS_DELETE_STATUS") == 0)
after_delete_filter = df_clean.count()
rows_removed_delete = before_delete_filter - after_delete_filter

print(f"Soft-delete filter:")
print(f"  Before : {before_delete_filter:,}")
print(f"  After  : {after_delete_filter:,}")
print(f"  Removed: {rows_removed_delete:,}")

StatementMeta(, a92cb0f9-a6af-4b67-8c11-7f99f7dfaabc, 39, Finished, Available, Finished, False)

Soft-delete filter:
  Before : 9,875,844
  After  : 9,875,708
  Removed: 136


---
## 4. Fix Data Types

### Why we are doing this

`SCHOOL_YEAR` is stored as `double` (e.g. `2024.0` instead of `2024`). This is a data type error from the source system or ETL pipeline. It causes three problems:

1. **Join failures** — if dimension tables store `SCHOOL_YEAR` as `int`, a double-to-int join may silently drop rows or produce unexpected matches
2. **Display issues** — `2024.0` in reports and charts looks unprofessional and confusing
3. **Floating-point risk** — `2024.0 == 2024` is true in most cases, but floating-point arithmetic can introduce subtle bugs (e.g. `2024.0000001` ≠ `2024`)

**Action:** Cast `SCHOOL_YEAR` to `IntegerType`. No data loss since school years are whole numbers.

In [38]:
# ============================================================
# 4.1 — Cast SCHOOL_YEAR from double to integer
# ============================================================
print(f"SCHOOL_YEAR dtype before: {dict(df_clean.dtypes)['SCHOOL_YEAR']}")

df_clean = df_clean.withColumn("SCHOOL_YEAR", F.col("SCHOOL_YEAR").cast(IntegerType()))

print(f"SCHOOL_YEAR dtype after : {dict(df_clean.dtypes)['SCHOOL_YEAR']}")

# Verify no NULLs were introduced by the cast (would happen if a value isn't a valid int)
null_sy = df_clean.filter(F.col("SCHOOL_YEAR").isNull()).count()
if null_sy == 0:
    print("✔ No NULL values introduced by cast.")
else:
    print(f"⚠ {null_sy:,} NULL values introduced — some SCHOOL_YEAR values couldn't be cast to int.")

StatementMeta(, a92cb0f9-a6af-4b67-8c11-7f99f7dfaabc, 40, Finished, Available, Finished, False)

SCHOOL_YEAR dtype before: double
SCHOOL_YEAR dtype after : int
✔ No NULL values introduced by cast.


In [39]:
# ============================================================
# 4.2 — Verify SCHOOL_YEAR range makes sense
# ============================================================
sy_range = df_clean.agg(
    F.min("SCHOOL_YEAR").alias("min_year"),
    F.max("SCHOOL_YEAR").alias("max_year"),
    F.countDistinct("SCHOOL_YEAR").alias("distinct_years")
).collect()[0]

print(f"SCHOOL_YEAR range: {sy_range['min_year']} to {sy_range['max_year']} ({sy_range['distinct_years']} distinct years)")

# Show all distinct school years
print("\nAll school years in data:")
display(
    df_clean
    .groupBy("SCHOOL_YEAR")
    .agg(F.count("*").alias("records"), F.countDistinct("STUDENT_KEY").alias("students"))
    .orderBy("SCHOOL_YEAR")
)

StatementMeta(, a92cb0f9-a6af-4b67-8c11-7f99f7dfaabc, 41, Finished, Available, Finished, False)

SCHOOL_YEAR range: 2001 to 2027 (27 distinct years)

All school years in data:


SynapseWidget(Synapse.DataFrame, e502cc94-690b-4742-bc2c-85cfcdf0fcdc)

---
## 5. Handle Sentinel Dates

### Why we are doing this

The source system uses `9999-12-31 23:59:59.997` in `SOURCE_ENROLLMENT_EXIT_DATE` to indicate **"this student has not yet exited"** — they are still enrolled. The EDA found **2,718,858 rows (13.77%)** with this sentinel value.

If left as-is, any calculation involving exit dates (enrollment duration, days enrolled, tenure features) will produce garbage values — for example, `9999-12-31` minus `2024-08-23` = **2,913,253 days**, which would dominate any statistical analysis.

**Action:** Replace sentinel dates with `NULL`. This is semantically correct — NULL means "unknown/not yet occurred" — and PySpark/SQL functions naturally handle NULLs in aggregations (they're excluded from AVG, MIN, MAX, etc.). Downstream code that needs a concrete date can use `COALESCE(exit_date, current_date())` explicitly.

In [40]:
# ============================================================
# 5.1 — Count sentinel dates before fix
# ============================================================
date_cols_to_check = [
    "SOURCE_ENROLLMENT_EXIT_DATE",
    "ENROLLMENT_ANNUALIZED_EXIT_DATE"
]

print("Sentinel date counts (year = 9999) before cleaning:")
for col_name in date_cols_to_check:
    cnt = df_clean.filter(F.year(F.col(col_name)) == 9999).count()
    pct = round(cnt / df_clean.count() * 100, 2)
    print(f"  {col_name}: {cnt:,} ({pct}%)")

StatementMeta(, a92cb0f9-a6af-4b67-8c11-7f99f7dfaabc, 42, Finished, Available, Finished, False)

Sentinel date counts (year = 9999) before cleaning:
  SOURCE_ENROLLMENT_EXIT_DATE: 1,359,429 (13.77%)
  ENROLLMENT_ANNUALIZED_EXIT_DATE: 0 (0.0%)


In [41]:
# ============================================================
# 5.2 — Replace sentinel dates with NULL
# ============================================================
for col_name in date_cols_to_check:
    df_clean = df_clean.withColumn(
        col_name,
        F.when(F.year(F.col(col_name)) == 9999, F.lit(None))
         .otherwise(F.col(col_name))
    )

# Also fix the corresponding date KEY columns if they encode the sentinel
# EXIT_CALENDAR_DATE_KEY and EXIT_SCHOOL_DATE_KEY may have sentinel values too
# We NULL them when the corresponding date is NULL
df_clean = df_clean.withColumn(
    "EXIT_CALENDAR_DATE_KEY",
    F.when(F.col("SOURCE_ENROLLMENT_EXIT_DATE").isNull(), F.lit(None))
     .otherwise(F.col("EXIT_CALENDAR_DATE_KEY"))
).withColumn(
    "EXIT_SCHOOL_DATE_KEY",
    F.when(F.col("SOURCE_ENROLLMENT_EXIT_DATE").isNull(), F.lit(None))
     .otherwise(F.col("EXIT_SCHOOL_DATE_KEY"))
)

print("✔ Sentinel dates replaced with NULL.")

# Verify
print("\nSentinel date counts after cleaning:")
for col_name in date_cols_to_check:
    cnt = df_clean.filter(F.year(F.col(col_name)) == 9999).count()
    print(f"  {col_name}: {cnt:,}")

StatementMeta(, a92cb0f9-a6af-4b67-8c11-7f99f7dfaabc, 43, Finished, Available, Finished, False)

✔ Sentinel dates replaced with NULL.

Sentinel date counts after cleaning:
  SOURCE_ENROLLMENT_EXIT_DATE: 0
  ENROLLMENT_ANNUALIZED_EXIT_DATE: 0


---
## 6. Remove Logically Invalid Records

### Why we are doing this

The EDA found **10 rows** where `SOURCE_ENROLLMENT_ENTRY_DATE > SOURCE_ENROLLMENT_EXIT_DATE` (excluding sentinel dates). This is logically impossible — a student cannot exit a school before they entered it. These are data entry errors or ETL corruption.

10 rows out of ~9.8M is negligible in volume, but leaving them in means:
- Negative enrollment duration values that break statistical calculations
- Potential for these records to propagate into aggregated features and silently skew results

**Action:** Remove these rows. They cannot be corrected without access to the source system.

In [42]:
# ============================================================
# 6.1 — Identify and remove logically invalid date records
# Only check rows where both dates are non-NULL (sentinel dates
# were already converted to NULL in step 5).
# ============================================================
before_date_fix = df_clean.count()

invalid_date_records = df_clean.filter(
    (F.col("SOURCE_ENROLLMENT_ENTRY_DATE").isNotNull()) &
    (F.col("SOURCE_ENROLLMENT_EXIT_DATE").isNotNull()) &
    (F.col("SOURCE_ENROLLMENT_ENTRY_DATE") > F.col("SOURCE_ENROLLMENT_EXIT_DATE"))
)

invalid_count = invalid_date_records.count()
print(f"Records with entry date > exit date: {invalid_count:,}")

if invalid_count > 0:
    print("\nSample of invalid records being removed:")
    display(
        invalid_date_records.select(
            "ENROLLMENT_ANNUALIZED_KEY", "STUDENT_KEY", "SCHOOL_KEY",
            "SCHOOL_YEAR", "SOURCE_ENROLLMENT_ENTRY_DATE",
            "SOURCE_ENROLLMENT_EXIT_DATE", "ENROLLMENT_DAYS"
        ).limit(10)
    )

# Remove invalid records
df_clean = df_clean.filter(
    # Keep rows where: dates are valid OR exit date is NULL (still enrolled)
    (F.col("SOURCE_ENROLLMENT_EXIT_DATE").isNull()) |
    (F.col("SOURCE_ENROLLMENT_ENTRY_DATE") <= F.col("SOURCE_ENROLLMENT_EXIT_DATE"))
)

after_date_fix = df_clean.count()
rows_removed_invalid_dates = before_date_fix - after_date_fix
print(f"\nRemoved {rows_removed_invalid_dates:,} invalid-date records.")

StatementMeta(, a92cb0f9-a6af-4b67-8c11-7f99f7dfaabc, 44, Finished, Available, Finished, False)

Records with entry date > exit date: 5

Sample of invalid records being removed:


SynapseWidget(Synapse.DataFrame, 6d1668a1-fa5d-4e7f-8c0b-e6bae704d247)


Removed 5 invalid-date records.


---
## 7. Drop Near-Empty Columns

### Why we are doing this

Two columns have fill rates so low that they carry essentially no usable information:

- **`ENTRY_REASON`**: 97.33% NULL (only ~527,000 out of ~19.7M rows had a value, and the only value seen was "Year End Processing")
- **`EXIT_REASON`**: 89.24% NULL (only ~2.1M rows had values, and those were numeric codes with no description)

These columns are not useful for ML features — the entry/exit *codes* (`ENTRY_CODE`, `EXIT_CODE`) already capture the same information with much higher fill rates and clearer taxonomy. Keeping these columns wastes storage, memory, and compute in downstream operations.

**Action:** Drop both columns.

In [43]:
# ============================================================
# 7.1 — Verify fill rates before dropping
# ============================================================
current_count = df_clean.count()
cols_to_drop = ["ENTRY_REASON", "EXIT_REASON"]

print("Fill rates for columns being dropped:")
for col_name in cols_to_drop:
    non_null = df_clean.filter(F.col(col_name).isNotNull()).count()
    fill_pct = round(non_null / current_count * 100, 2)
    print(f"  {col_name}: {fill_pct}% filled ({non_null:,} / {current_count:,})")

StatementMeta(, a92cb0f9-a6af-4b67-8c11-7f99f7dfaabc, 45, Finished, Available, Finished, False)

Fill rates for columns being dropped:
  ENTRY_REASON: 2.67% filled (263,706 / 9,875,703)
  EXIT_REASON: 10.76% filled (1,063,118 / 9,875,703)


In [44]:
# ============================================================
# 7.2 — Drop near-empty columns
# ============================================================
df_clean = df_clean.drop(*cols_to_drop)

print(f"✔ Dropped {len(cols_to_drop)} columns: {cols_to_drop}")
print(f"  Columns remaining: {len(df_clean.columns)}")

StatementMeta(, a92cb0f9-a6af-4b67-8c11-7f99f7dfaabc, 46, Finished, Available, Finished, False)

✔ Dropped 2 columns: ['ENTRY_REASON', 'EXIT_REASON']
  Columns remaining: 27


---
## 8. Add Cleaning Metadata & Utility Columns

### Why we are doing this

Rather than deleting records that need special handling (future years, zero-day enrollments), we add **flag columns** that downstream consumers can filter on. This is the production-grade approach because:

1. **Future school year records (SY ≥ 2027):** These are pre-enrolled students. They are real records, but they should NOT be used in ML training data (that would be target leakage). However, they ARE useful for validation and for the operations team. Instead of deleting them, we flag them.

2. **Zero-day enrollment records (ENROLLMENT_DAYS = 0):** 69.65% of all records have zero enrollment days. This is because `fact_enrollment_annualized` creates an annualized row for each school year a student is associated with, even if the school year hasn't started yet (or if the student's actual attendance falls in a different window). For the 20th-day headcount (our ML target), we need to filter to records where the student actually attended — but other analyses (e.g., pre-enrollment forecasting) legitimately need these rows.

3. **Entry grade flag:** Kindergarten and 9th grade are "entry grades" that behave fundamentally differently from grade-to-grade transitions and require separate ML models. Flagging them here saves repeated logic downstream.

4. **Enrollment duration:** A clean, computed duration in days — using `COALESCE` for currently-enrolled students — avoids every downstream consumer having to repeat sentinel-date logic.


In [45]:
# ============================================================
# 8.1 — Add flag columns (with capped enrollment duration)
# ============================================================

CURRENT_YEAR = 2026  # Update this each school year

# For annualized records, any duration much above a school year is suspicious.
# We conservatively cap at 366 days to prevent extreme outliers (e.g., 7,901 days).
MAX_ENROLLMENT_DURATION_DAYS = 366

df_clean = (
    df_clean

    # Flag: Is this a future school year? (pre-enrollment, not actual)
    .withColumn(
        "IS_FUTURE_SCHOOL_YEAR",
        F.when(F.col("SCHOOL_YEAR") > CURRENT_YEAR, F.lit(True))
         .otherwise(F.lit(False))
    )

    # Flag: Is this an entry grade? (K or 9 — hardest to forecast)
    .withColumn(
        "IS_ENTRY_GRADE",
        F.when(
            F.col("ENTRY_GRADE_LEVEL").cast("string").isin("K", "9"),
            F.lit(True)
        ).otherwise(F.lit(False))
    )

    # Flag: Zero-day enrollment (annualized window, not actual attendance)
    .withColumn(
        "IS_ZERO_DAY_ENROLLMENT",
        F.when(F.col("ENROLLMENT_DAYS") == 0, F.lit(True))
         .otherwise(F.lit(False))
    )

    # Flag: Is the student currently enrolled?
    .withColumn(
        "IS_CURRENTLY_ENROLLED",
        F.when(F.col("CURRENT_ENROLLMENT_INDICATOR") == "Yes", F.lit(True))
         .otherwise(F.lit(False))
    )

    # Computed: raw enrollment duration in days
    # For currently enrolled (NULL exit date), use current date as proxy
    .withColumn(
        "ENROLLMENT_DURATION_DAYS",
        F.datediff(
            F.coalesce(F.col("SOURCE_ENROLLMENT_EXIT_DATE"), F.current_date()),
            F.col("SOURCE_ENROLLMENT_ENTRY_DATE")
        )
    )

    # Cap extreme durations to avoid pathological values (e.g., > 21 years)
    .withColumn(
        "ENROLLMENT_DURATION_DAYS",
        F.when(F.col("ENROLLMENT_DURATION_DAYS") > MAX_ENROLLMENT_DURATION_DAYS,
               F.lit(MAX_ENROLLMENT_DURATION_DAYS))
         .otherwise(F.col("ENROLLMENT_DURATION_DAYS"))
    )
)

print("✔ Added 5 utility columns (with capped ENROLLMENT_DURATION_DAYS):")
print("  • IS_FUTURE_SCHOOL_YEAR   — True if SCHOOL_YEAR > 2026")
print("  • IS_ENTRY_GRADE          — True if grade is K or 9")
print("  • IS_ZERO_DAY_ENROLLMENT  — True if ENROLLMENT_DAYS = 0")
print("  • IS_CURRENTLY_ENROLLED   — True if CURRENT_ENROLLMENT_INDICATOR = 'Yes'")
print("  • ENROLLMENT_DURATION_DAYS — Computed duration, capped at 366 days")

StatementMeta(, a92cb0f9-a6af-4b67-8c11-7f99f7dfaabc, 47, Finished, Available, Finished, False)

✔ Added 5 utility columns (with capped ENROLLMENT_DURATION_DAYS):
  • IS_FUTURE_SCHOOL_YEAR   — True if SCHOOL_YEAR > 2026
  • IS_ENTRY_GRADE          — True if grade is K or 9
  • IS_ZERO_DAY_ENROLLMENT  — True if ENROLLMENT_DAYS = 0
  • IS_CURRENTLY_ENROLLED   — True if CURRENT_ENROLLMENT_INDICATOR = 'Yes'
  • ENROLLMENT_DURATION_DAYS — Computed duration, capped at 366 days


In [46]:
# ============================================================
# 8.2 — Validate flag distributions
# ============================================================
flag_cols = [
    "IS_FUTURE_SCHOOL_YEAR",
    "IS_ENTRY_GRADE",
    "IS_ZERO_DAY_ENROLLMENT",
    "IS_CURRENTLY_ENROLLED"
]

total = df_clean.count()
print(f"Flag column distributions (total rows: {total:,}):\n")

for flag in flag_cols:
    true_count = df_clean.filter(F.col(flag) == True).count()
    pct = round(true_count / total * 100, 2)
    print(f"  {flag:35s}: {true_count:>12,} True ({pct:>6}%)")

StatementMeta(, a92cb0f9-a6af-4b67-8c11-7f99f7dfaabc, 48, Finished, Available, Finished, False)

Flag column distributions (total rows: 9,875,703):

  IS_FUTURE_SCHOOL_YEAR              :      331,776 True (  3.36%)
  IS_ENTRY_GRADE                     :    3,499,193 True ( 35.43%)
  IS_ZERO_DAY_ENROLLMENT             :    6,878,469 True ( 69.65%)
  IS_CURRENTLY_ENROLLED              :      331,768 True (  3.36%)


In [47]:
# ============================================================
# 8.3 — Validate ENROLLMENT_DURATION_DAYS
# ============================================================
duration_stats = df_clean.agg(
    F.min("ENROLLMENT_DURATION_DAYS").alias("min_days"),
    F.max("ENROLLMENT_DURATION_DAYS").alias("max_days"),
    F.avg("ENROLLMENT_DURATION_DAYS").alias("avg_days"),
    F.percentile_approx("ENROLLMENT_DURATION_DAYS", 0.5).alias("median_days"),
    F.count(F.when(F.col("ENROLLMENT_DURATION_DAYS") < 0, 1)).alias("negative_count")
).collect()[0]

print("ENROLLMENT_DURATION_DAYS (computed, sentinel-safe):")
print(f"  Min    : {duration_stats['min_days']}")
print(f"  Max    : {duration_stats['max_days']}")
print(f"  Mean   : {duration_stats['avg_days']:.1f}")
print(f"  Median : {duration_stats['median_days']}")
print(f"  Negatives: {duration_stats['negative_count']:,}")

if duration_stats['max_days'] and duration_stats['max_days'] > 3650:
    print(f"\n  ⚠ Max duration is {duration_stats['max_days']:,} days ({duration_stats['max_days']//365} years).")
    print("    This may indicate SOURCE_ENROLLMENT_ENTRY_DATE was set far in the past.")
    print("    Inspect extreme values to determine if capping is needed.")

StatementMeta(, a92cb0f9-a6af-4b67-8c11-7f99f7dfaabc, 49, Finished, Available, Finished, False)

ENROLLMENT_DURATION_DAYS (computed, sentinel-safe):
  Min    : -271
  Max    : 366
  Mean   : 327.1
  Median : 366
  Negatives: 4


---
## 9. Final Validation & Write Clean Table

In [48]:
# ============================================================
# 9.1 — Final schema review
# ============================================================
print("Clean table schema:")
df_clean.printSchema()

StatementMeta(, a92cb0f9-a6af-4b67-8c11-7f99f7dfaabc, 50, Finished, Available, Finished, False)

Clean table schema:
root
 |-- ENROLLMENT_ANNUALIZED_KEY: long (nullable = true)
 |-- SCHOOL_YEAR: integer (nullable = true)
 |-- STUDENT_KEY: long (nullable = true)
 |-- SCHOOL_KEY: long (nullable = true)
 |-- ENROLLMENT_TYPE: string (nullable = true)
 |-- CURRENT_ENROLLMENT_INDICATOR: string (nullable = true)
 |-- ENTRY_CALENDAR_DATE_KEY: long (nullable = true)
 |-- ENTRY_SCHOOL_DATE_KEY: long (nullable = true)
 |-- EXIT_CALENDAR_DATE_KEY: long (nullable = true)
 |-- EXIT_SCHOOL_DATE_KEY: long (nullable = true)
 |-- ENTRY_CODE: string (nullable = true)
 |-- ENTRY_CODE_DESCRIPTION: string (nullable = true)
 |-- EXIT_CODE: string (nullable = true)
 |-- EXIT_CODE_DESCRIPTION: string (nullable = true)
 |-- ENTRY_GRADE_LEVEL: string (nullable = true)
 |-- EXIT_GRADE_LEVEL: string (nullable = true)
 |-- SOURCE_ENROLLMENT_ENTRY_DATE: timestamp (nullable = true)
 |-- SOURCE_ENROLLMENT_EXIT_DATE: timestamp (nullable = true)
 |-- ENROLLMENT_DAYS: integer (nullable = true)
 |-- SCHOOL_IN_SESSION

In [49]:
# ============================================================
# 9.2 — Final quality assertions
# These are hard checks. If any fails, the clean table should
# NOT be written — there's a data issue to investigate.
# ============================================================
final_count = df_clean.count()
final_keys = df_clean.select("ENROLLMENT_ANNUALIZED_KEY").distinct().count()

checks = []

# Check 1: Primary key uniqueness
pk_ok = (final_count == final_keys)
checks.append(("PK Uniqueness", pk_ok, f"{final_keys:,} keys vs {final_count:,} rows"))

# Check 2: No SYS_DELETE_STATUS = 1
delete_count = df_clean.filter(F.col("SYS_DELETE_STATUS") == 1).count()
delete_ok = (delete_count == 0)
checks.append(("No soft-deletes", delete_ok, f"{delete_count} deleted rows found"))

# Check 3: SCHOOL_YEAR is integer
sy_type = dict(df_clean.dtypes)["SCHOOL_YEAR"]
type_ok = (sy_type == "int")
checks.append(("SCHOOL_YEAR is int", type_ok, f"Type is {sy_type}"))

# Check 4: No sentinel dates remain
sentinel_count = df_clean.filter(F.year("SOURCE_ENROLLMENT_EXIT_DATE") == 9999).count()
sentinel_ok = (sentinel_count == 0)
checks.append(("No sentinel dates", sentinel_ok, f"{sentinel_count} sentinel dates found"))

# Check 5: No entry > exit date (excluding NULLs)
bad_dates = df_clean.filter(
    (F.col("SOURCE_ENROLLMENT_EXIT_DATE").isNotNull()) &
    (F.col("SOURCE_ENROLLMENT_ENTRY_DATE") > F.col("SOURCE_ENROLLMENT_EXIT_DATE"))
).count()
dates_ok = (bad_dates == 0)
checks.append(("No invalid date ranges", dates_ok, f"{bad_dates} bad date rows"))

# Check 6: ENTRY_REASON and EXIT_REASON dropped
dropped_ok = ("ENTRY_REASON" not in df_clean.columns and "EXIT_REASON" not in df_clean.columns)
checks.append(("Sparse columns dropped", dropped_ok, f"Columns: {df_clean.columns}"))

# Print results
all_passed = True
print("Final quality checks:")
print("─" * 60)
for name, passed, detail in checks:
    status = "✔ PASS" if passed else "✘ FAIL"
    print(f"  {status}  {name}")
    if not passed:
        print(f"         Detail: {detail}")
        all_passed = False
print("─" * 60)

if all_passed:
    print("\n✔ ALL CHECKS PASSED. Safe to write clean table.")
else:
    print("\n✘ SOME CHECKS FAILED. Investigate before writing.")

StatementMeta(, a92cb0f9-a6af-4b67-8c11-7f99f7dfaabc, 51, Finished, Available, Finished, False)

Final quality checks:
────────────────────────────────────────────────────────────
  ✔ PASS  PK Uniqueness
  ✔ PASS  No soft-deletes
  ✔ PASS  SCHOOL_YEAR is int
  ✔ PASS  No sentinel dates
  ✔ PASS  No invalid date ranges
  ✔ PASS  Sparse columns dropped
────────────────────────────────────────────────────────────

✔ ALL CHECKS PASSED. Safe to write clean table.


In [50]:
# ============================================================
# 9.3 — Cleaning audit summary
# ============================================================
print("="*65)
print("   CLEANING AUDIT SUMMARY")
print("="*65)
print(f"")
print(f"  INPUT")
print(f"  {'Raw table':40s}: {RAW_TABLE}")
print(f"  {'Raw row count':40s}: {RAW_ROW_COUNT:,}")
print(f"  {'Raw column count':40s}: {RAW_COL_COUNT}")
print(f"")
print(f"  ROWS REMOVED")
print(f"  {'Exact duplicates (ETL 2x load)':40s}: {rows_removed_dedup:,}")
print(f"  {'Soft-deleted (SYS_DELETE_STATUS=1)':40s}: {rows_removed_delete:,}")
print(f"  {'Invalid dates (entry > exit)':40s}: {rows_removed_invalid_dates:,}")
total_removed = rows_removed_dedup + rows_removed_delete + rows_removed_invalid_dates
print(f"  {'─'*40}")
print(f"  {'TOTAL REMOVED':40s}: {total_removed:,} ({total_removed / RAW_ROW_COUNT * 100:.1f}%)")
print(f"")
print(f"  COLUMNS CHANGED")
print(f"  {'Dropped':40s}: ENTRY_REASON, EXIT_REASON")
print(f"  {'Type-fixed':40s}: SCHOOL_YEAR (double → int)")
print(f"  {'Sentinel dates → NULL':40s}: SOURCE_ENROLLMENT_EXIT_DATE,")
print(f"  {'':40s}  ENROLLMENT_ANNUALIZED_EXIT_DATE")
print(f"  {'Added':40s}: IS_FUTURE_SCHOOL_YEAR, IS_ENTRY_GRADE,")
print(f"  {'':40s}  IS_ZERO_DAY_ENROLLMENT, IS_CURRENTLY_ENROLLED,")
print(f"  {'':40s}  ENROLLMENT_DURATION_DAYS")
print(f"")
print(f"  OUTPUT")
print(f"  {'Clean row count':40s}: {final_count:,}")
print(f"  {'Clean column count':40s}: {len(df_clean.columns)}")
print(f"  {'Unique students':40s}: {df_clean.select('STUDENT_KEY').distinct().count():,}")
print(f"  {'Unique schools':40s}: {df_clean.select('SCHOOL_KEY').distinct().count():,}")
print(f"  {'School year range':40s}: {sy_range['min_year']}–{sy_range['max_year']}")
print("="*65)

StatementMeta(, a92cb0f9-a6af-4b67-8c11-7f99f7dfaabc, 52, Finished, Available, Finished, False)

   CLEANING AUDIT SUMMARY

  INPUT
  Raw table                               : fact_enrollment_annualized
  Raw row count                           : 19,751,688
  Raw column count                        : 29

  ROWS REMOVED
  Exact duplicates (ETL 2x load)          : 9,875,844
  Soft-deleted (SYS_DELETE_STATUS=1)      : 136
  Invalid dates (entry > exit)            : 5
  ────────────────────────────────────────
  TOTAL REMOVED                           : 9,875,985 (50.0%)

  COLUMNS CHANGED
  Dropped                                 : ENTRY_REASON, EXIT_REASON
  Type-fixed                              : SCHOOL_YEAR (double → int)
  Sentinel dates → NULL                   : SOURCE_ENROLLMENT_EXIT_DATE,
                                            ENROLLMENT_ANNUALIZED_EXIT_DATE
  Added                                   : IS_FUTURE_SCHOOL_YEAR, IS_ENTRY_GRADE,
                                            IS_ZERO_DAY_ENROLLMENT, IS_CURRENTLY_ENROLLED,
                                        

In [51]:
# ============================================================
# 9.4 — Write clean table to Lakehouse
# ============================================================
CLEAN_TABLE = "fact_enrollment_annualized_clean"

if all_passed:
    df_clean.write.mode("overwrite").saveAsTable(CLEAN_TABLE)
    
    # Verify write
    verify_count = spark.read.table(CLEAN_TABLE).count()
    print(f"✔ Clean table written to Lakehouse: '{CLEAN_TABLE}'")
    print(f"  Verified row count: {verify_count:,}")
else:
    print("✘ Skipping write — quality checks did not pass.")
    print("  Fix the failing checks above, then re-run this cell.")

StatementMeta(, a92cb0f9-a6af-4b67-8c11-7f99f7dfaabc, 53, Finished, Available, Finished, False)

✔ Clean table written to Lakehouse: 'fact_enrollment_annualized_clean'
  Verified row count: 9,875,703


---
## 10. Post-Cleaning Verification

### Quick sanity checks on the clean table to confirm it makes sense.

In [52]:
# ============================================================
# 10.1 — Records-per-student ratio should now be ~1.0-1.2
# (Was uniformly ~2.0 in raw data due to ETL duplication)
# ============================================================
df_verify = spark.read.table(CLEAN_TABLE)

rps_check = (
    df_verify
    .groupBy("SCHOOL_YEAR")
    .agg(
        F.count("*").alias("records"),
        F.countDistinct("STUDENT_KEY").alias("students")
    )
    .withColumn("records_per_student", F.round(F.col("records") / F.col("students"), 3))
    .orderBy("SCHOOL_YEAR")
)

print("Records per student by year (post-cleaning):")
print("(Should be ~1.0-1.3 — was ~2.0 before dedup)\n")
display(rps_check)

StatementMeta(, a92cb0f9-a6af-4b67-8c11-7f99f7dfaabc, 54, Finished, Available, Finished, False)

Records per student by year (post-cleaning):
(Should be ~1.0-1.3 — was ~2.0 before dedup)



SynapseWidget(Synapse.DataFrame, 6a2a5420-78c2-45dd-bf23-8538e3335e10)

In [53]:
# ============================================================
# 10.2 — Enrollment counts should now look realistic
# CPS has ~316,000 students in 2025-26 per domain knowledge.
# ============================================================
enrollment_trend = (
    df_verify
    .filter(F.col("IS_FUTURE_SCHOOL_YEAR") == False)
    .groupBy("SCHOOL_YEAR")
    .agg(F.countDistinct("STUDENT_KEY").alias("unique_students"))
    .orderBy("SCHOOL_YEAR")
)

print("Unique student enrollment by year (excluding future years):")
print("(Reference: CPS had ~438K in 2002-03 and ~316K in 2025-26)\n")
display(enrollment_trend)

StatementMeta(, a92cb0f9-a6af-4b67-8c11-7f99f7dfaabc, 55, Finished, Available, Finished, False)

Unique student enrollment by year (excluding future years):
(Reference: CPS had ~438K in 2002-03 and ~316K in 2025-26)



SynapseWidget(Synapse.DataFrame, 0c4cd3b6-2c27-499c-8c88-b4ca16a7a25a)

In [54]:
# ============================================================
# 10.3 — Preview the clean table
# ============================================================
print(f"Clean table: {CLEAN_TABLE}")
print(f"Columns: {df_verify.columns}\n")
display(df_verify.limit(10))

StatementMeta(, a92cb0f9-a6af-4b67-8c11-7f99f7dfaabc, 56, Finished, Available, Finished, False)

Clean table: fact_enrollment_annualized_clean
Columns: ['ENROLLMENT_ANNUALIZED_KEY', 'SCHOOL_YEAR', 'STUDENT_KEY', 'SCHOOL_KEY', 'ENROLLMENT_TYPE', 'CURRENT_ENROLLMENT_INDICATOR', 'ENTRY_CALENDAR_DATE_KEY', 'ENTRY_SCHOOL_DATE_KEY', 'EXIT_CALENDAR_DATE_KEY', 'EXIT_SCHOOL_DATE_KEY', 'ENTRY_CODE', 'ENTRY_CODE_DESCRIPTION', 'EXIT_CODE', 'EXIT_CODE_DESCRIPTION', 'ENTRY_GRADE_LEVEL', 'EXIT_GRADE_LEVEL', 'SOURCE_ENROLLMENT_ENTRY_DATE', 'SOURCE_ENROLLMENT_EXIT_DATE', 'ENROLLMENT_DAYS', 'SCHOOL_IN_SESSION_INDICATOR', 'SOURCE_SURROGATE_KEY_ENTRY', 'SOURCE_SURROGATE_KEY_EXIT', 'ENROLLMENT_ANNUALIZED_START_DATE', 'ENROLLMENT_ANNUALIZED_EXIT_DATE', 'SYS_DELETE_STATUS', 'CREATED_TS', 'LAST_UPDATED_TS', 'IS_FUTURE_SCHOOL_YEAR', 'IS_ENTRY_GRADE', 'IS_ZERO_DAY_ENROLLMENT', 'IS_CURRENTLY_ENROLLED', 'ENROLLMENT_DURATION_DAYS']



SynapseWidget(Synapse.DataFrame, 94d323c7-dcf1-4e10-a1b9-a907b394474f)

In [55]:
# ============================================================
# 10.4 — Usage guide for downstream notebooks
# ============================================================
print("""
╔══════════════════════════════════════════════════════════════╗
║  HOW TO USE THE CLEAN TABLE IN DOWNSTREAM NOTEBOOKS        ║
╠══════════════════════════════════════════════════════════════╣
║                                                            ║
║  # Load clean table                                        ║
║  df = spark.read.table("fact_enrollment_annualized_clean") ║
║                                                            ║
║  # For ML training data (exclude future years):            ║
║  df_train = df.filter(                                     ║
║      (F.col("IS_FUTURE_SCHOOL_YEAR") == False)             ║
║  )                                                         ║
║                                                            ║
║  # For 20th-day headcount (exclude zero-day records):      ║
║  df_actual = df.filter(                                    ║
║      (F.col("IS_FUTURE_SCHOOL_YEAR") == False) &           ║
║      (F.col("IS_ZERO_DAY_ENROLLMENT") == False)            ║
║  )                                                         ║
║                                                            ║
║  # For entry grade models only:                            ║
║  df_entry = df.filter(F.col("IS_ENTRY_GRADE") == True)     ║
║                                                            ║
║  # For currently enrolled snapshot:                        ║
║  df_current = df.filter(                                   ║
║      F.col("IS_CURRENTLY_ENROLLED") == True                ║
║  )                                                         ║
║                                                            ║
╚══════════════════════════════════════════════════════════════╝
""")

StatementMeta(, a92cb0f9-a6af-4b67-8c11-7f99f7dfaabc, 57, Finished, Available, Finished, False)


╔══════════════════════════════════════════════════════════════╗
║  HOW TO USE THE CLEAN TABLE IN DOWNSTREAM NOTEBOOKS        ║
╠══════════════════════════════════════════════════════════════╣
║                                                            ║
║  # Load clean table                                        ║
║  df = spark.read.table("fact_enrollment_annualized_clean") ║
║                                                            ║
║  # For ML training data (exclude future years):            ║
║  df_train = df.filter(                                     ║
║      (F.col("IS_FUTURE_SCHOOL_YEAR") == False)             ║
║  )                                                         ║
║                                                            ║
║  # For 20th-day headcount (exclude zero-day records):      ║
║  df_actual = df.filter(                                    ║
║      (F.col("IS_FUTURE_SCHOOL_YEAR") == False) &           ║
║      (F.col("IS_ZERO_DAY_ENROLLMENT") == False) 